In [ ]:
!pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib icalendar pytz python-dateutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 5.4 MB/s eta 0:00:00


In [ ]:
# Device login (copy/paste code) with both scopes
!gcloud auth application-default login --no-launch-browser --scopes=https://www.googleapis.com/auth/cloud-platform,https://www.googleapis.com/auth/calendar

# Attach your quota project and make sure Calendar API is enabled
!gcloud auth application-default set-quota-project california-477322
!gcloud services enable calendar-json.googleapis.com --project=california-477322


Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=L4YczF4ydI37zKkCJhgdKKgRWxzCs4&prompt=consent&token_usage=remote&access_type=offline&code_challenge=QLtQdDyzawYS3tiTOy8N2MtekJcWWMXiD6x7_aCQG_A&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0Ab32j90MLNpeNPvTZrSTGDZTdhlp10TR_J_ha6VbkoLAiAyaAn8HJcjeCNK9H-Ph3CVjFg

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Cannot find a quota project to add to ADC. You might receive a "quota exce

In [ ]:
import google.auth
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

SCOPES = ["https://www.googleapis.com/auth/calendar"]
creds, _ = google.auth.default(scopes=SCOPES)
creds.refresh(Request())  # ensure token is fresh & scoped

service = build("calendar", "v3", credentials=creds, cache_discovery=False)

# sanity check (should not 403)
print("Primary calendar:", service.calendarList().get(calendarId="primary").execute().get("summary"))

Primary calendar: leelaprasad.dammalapati@sjsu.edu


In [ ]:
from datetime import datetime, timedelta, date
from icalendar import Calendar
import pytz, json

ICS_PATH = "/content/email.ics"  # change if needed

def _pack(dt_obj, tzid):
    if isinstance(dt_obj, date) and not isinstance(dt_obj, datetime):
        return {"mode":"date","value":dt_obj.isoformat(),"tz":None}
    if isinstance(dt_obj, datetime):
        if dt_obj.tzinfo is None:
            dt_obj = (pytz.timezone(tzid) if tzid else pytz.UTC).localize(dt_obj)
        tzname = getattr(getattr(dt_obj,"tzinfo",None),"zone",None) or tzid or "UTC"
        return {"mode":"dateTime","value":dt_obj.isoformat(),"tz":tzname}
    raise ValueError("Bad time in ICS")

def parse_ics_event(path):
    with open(path,"rb") as f:
        cal = Calendar.from_ical(f.read())
    for comp in cal.walk("VEVENT"):
        summary = str(comp.get("summary") or "Untitled event")
        description = str(comp.get("description") or "")
        location = str(comp.get("location") or "")
        dtstart_prop = comp.get("dtstart")
        if not dtstart_prop: raise ValueError("Missing DTSTART")
        dtstart = comp.decoded("dtstart")
        dtend = comp.decoded("dtend", None)
        if dtend is None:
            dur = comp.get("duration")
            if dur: dtend = dtstart + dur.dt
            else:   dtend = (dtstart + timedelta(hours=1)) if isinstance(dtstart, datetime) else (dtstart + timedelta(days=1))
        tzid = getattr(dtstart_prop, "params", {}).get("TZID")
        s, e = _pack(dtstart, tzid), _pack(dtend, tzid)

        # attendees
        raw = comp.get("attendee"); attendees=[]
        if raw:
            raw = raw if isinstance(raw, list) else [raw]
            for a in raw:
                s2 = str(a)
                if "mailto:" in s2.lower():
                    email = s2.split(":",1)[1].strip()
                    if email: attendees.append({"email": email})
        # de-dupe
        seen=set(); uniq=[]
        for a in attendees:
            em=a["email"].lower()
            if em not in seen: uniq.append(a); seen.add(em)

        evt={"summary":summary,"description":description,"location":location}
        if s["mode"]=="dateTime":
            evt["start"]={"dateTime":s["value"],"timeZone":s["tz"]}
            evt["end"]  ={"dateTime":e["value"],"timeZone":e["tz"]}
        else:
            evt["start"]={"date":s["value"]}
            evt["end"]  ={"date":e["value"]}
        if uniq: evt["attendees"]=uniq
        return evt
    raise ValueError("No VEVENT in ICS")

# Use existing payload if present; otherwise parse
if "event_payload" not in globals():
    event_payload = parse_ics_event(ICS_PATH)
    print("Parsed ICS:\n", json.dumps(event_payload, indent=2))

created = service.events().insert(
    calendarId="primary",
    body=event_payload,
    sendUpdates="all"   # emails attendees
).execute()

print("Created:", created.get("summary"))
print("Open in Calendar:", created.get("htmlLink"))


Created: Q4 Planning Session
Open in Calendar: https://www.google.com/calendar/event?eid=Z2Ewdmc2MjlmODNlMnByOG1rMjRicGs0c2sgbGVlbGFwcmFzYWQuZGFtbWFsYXBhdGlAc2pzdS5lZHU
